# Run a test client

## Apply the feature store definitions

We'll mount the `feature_store.yaml` ConfigMap created by the Operator within a Kubernetes `Deployment` to run `feast apply`.

Then we create the client `Deployment` to apply the definitions, according to the [client.yaml](./client.yaml) manifest

We are going to emulate the `feast init -t postgres sample` command using Python code in an initContainer at client `Deployment` runtime. This is needed to mock the request of additional
parameters to configure the DB connection and also request the upload of example data to Postgres tables.

In [1]:
!kubectl apply -f client.yaml

serviceaccount/feast-example-client unchanged
deployment.apps/feast-example-client unchanged


Monitoring the log of the `Deployment` and checking DB for populated data.

In [1]:
!kubectl wait --for=condition=available deploy/feast-example-client --timeout=2m
# !kubectl logs deploy/feast-example-client -c feast-apply
!kubectl exec deploy/postgres -- psql -h localhost -U feast feast -c 'create database feast'

deployment.apps/feast-example-client condition met
ERROR:  database "feast" already exists
command terminated with exit code 1


Verify the client `feature_store.yaml`.

In [1]:
#!wget https://github.com/feast-dev/feast-credit-score-local-tutorial/archive/f43b44b.tar.gz
!wget https://github.com/tchughesiv/feast-credit-score-local-tutorial/archive/dev.tar.gz
!kubectl cp dev.tar.gz $(kubectl get pods -l app=feast-example-client -ojsonpath="{.items[*].metadata.name}"):/feast-init -c client
!kubectl exec deploy/feast-example-client -c client -- mkdir feast-credit-score-local-tutorial
!kubectl exec deploy/feast-example-client -c client -- tar vfx dev.tar.gz -C feast-credit-score-local-tutorial --strip-components 1
!kubectl cp feature_store.yaml $(kubectl get pods -l app=feast-example-client -ojsonpath="{.items[*].metadata.name}"):/feast-init/feast-credit-score-local-tutorial/feature_repo/ -c client
!kubectl exec deploy/feast-example-client -c client -- cat /feast-init/feast-credit-score-local-tutorial/feature_repo/feature_store.yaml
!kubectl exec deploy/feast-example-client -c client -- feast -c /feast-init/feast-credit-score-local-tutorial/feature_repo apply

--2024-12-15 15:20:01--  https://github.com/tchughesiv/feast-credit-score-local-tutorial/archive/dev.tar.gz
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/tchughesiv/feast-credit-score-local-tutorial/tar.gz/refs/heads/dev [following]
--2024-12-15 15:20:01--  https://codeload.github.com/tchughesiv/feast-credit-score-local-tutorial/tar.gz/refs/heads/dev
Resolving codeload.github.com (codeload.github.com)... 140.82.112.10
Connecting to codeload.github.com (codeload.github.com)|140.82.112.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/x-gzip]
Saving to: ‘dev.tar.gz’

dev.tar.gz              [      <=>           ]  45.52M  8.31MB/s    in 5.6s    

2024-12-15 15:20:07 (8.12 MB/s) - ‘dev.tar.gz’ saved [47734286]

mkdir: cannot create directory ‘feast-credit-score-local-tutorial’: 

In [3]:
!kubectl exec deploy/feast-example-client -c client -- feast -c /feast-init/feast-credit-score-local-tutorial/feature_repo materialize-incremental $(date -u +"%Y-%m-%dT%H:%M:%S")

0it [00:00, ?it/s]
Materializing 2 feature views to 2024-12-15 20:21:03+00:00 into the remote online store.

credit_history from 2024-12-13 23:27:15+00:00 to 2024-12-15 20:21:03+00:00:
0it [00:00, ?it/s]
zipcode_features from 2024-12-13 23:27:15+00:00 to 2024-12-15 20:21:03+00:00:


## Execute feast commands inside the client Pod

Finally, we run the full test suite from the client folder.

In [4]:
!kubectl exec deploy/feast-example-client -c client -- feast -c /feast-init/feast-credit-score-local-tutorial/feature_repo feature-views list
!kubectl exec deploy/feast-example-client -c client -- feast -c /feast-init/feast-credit-score-local-tutorial/feature_repo feature-views describe credit_history
!kubectl exec deploy/feast-example-client -c client -- feast -c /feast-init/feast-credit-score-local-tutorial/feature_repo entities list
!kubectl exec deploy/feast-example-client -c client -- feast -c /feast-init/feast-credit-score-local-tutorial/feature_repo data-sources list

NAME              ENTITIES     TYPE
credit_history    {'dob_ssn'}  FeatureView
zipcode_features  {'zipcode'}  FeatureView
total_debt_calc   {'dob_ssn'}  OnDemandFeatureView
spec:
  name: credit_history
  entities:
  - dob_ssn
  features:
  - name: credit_card_due
    valueType: INT64
  - name: mortgage_due
    valueType: INT64
  - name: student_loan_due
    valueType: INT64
  - name: vehicle_loan_due
    valueType: INT64
  - name: hard_pulls
    valueType: INT64
  - name: missed_payments_2y
    valueType: INT64
  - name: missed_payments_1y
    valueType: INT64
  - name: missed_payments_6m
    valueType: INT64
  - name: bankruptcies
    valueType: INT64
  ttl: 7776000s
  batchSource:
    type: BATCH_FILE
    timestampField: event_timestamp
    createdTimestampColumn: created_timestamp
    fileOptions:
      fileFormat:
        parquetFormat: {}
      uri: data/credit_history.parquet
    dataSourceClassType: feast.infra.offline_stores.file_source.FileSource
    name: Credit history
  onl

## Run test code inside the client Pod

Finally, we run the full test suite from the client folder.

In [ ]:
!kubectl exec deploy/feast-example-client -c client -- python3 -m venv --system-site-packages .venv
!kubectl exec deploy/feast-example-client -c client -- bash -c 'source .venv/bin/activate && pip install -r /feast-init/feast-credit-score-local-tutorial/requirements.txt'
!kubectl exec deploy/feast-example-client -c client -- bash -c 'source .venv/bin/activate && cd /feast-init/feast-credit-score-local-tutorial && python run.py'
!kubectl exec deploy/feast-example-client -c client -- bash -c 'source .venv/bin/activate && cd /feast-init/feast-credit-score-local-tutorial && streamlit run streamlit_app.py'


[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


## Run feast python code inside the client Pod

Finally, we run the full test suite from the client folder.

In [5]:
!kubectl exec deploy/feast-example-client -c client -- cat << EOF > blah.py
from pprint import pprint
from feast import FeatureStore

store = FeatureStore(repo_path=".")

feature_vector = store.get_online_features(
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
    ],
    entity_rows=[
        # {join_key: entity_value}
        {"driver_id": 1004},
        {"driver_id": 1005},
    ],
).to_dict()

pprint(feature_vector)
EOF

!kubectl exec deploy/feast-example-client -c client -- python blah.py

FileNotFoundError: [Errno 2] No such file or directory: 'feature_store.yaml'